# Duplicates and Sorting

*Identify repeated order keys, fill a missing quantity, and put retained records in a clear order.*


## Before You Run

In **Add files**, select `retail_sales_raw.csv` from this lesson's supplied assets.
Enter `assets/03_duplicates_and_sorting` for **Save folder**. The upload creates
missing folders and places the file at
`/home/pyodide/assets/03_duplicates_and_sorting/retail_sales_raw.csv`.

`/home/pyodide` is the browser's virtual filesystem, not a folder on your
computer. The upload folder is relative to `/home/pyodide`; Python relative
paths use the current directory returned by `os.getcwd()`. Keep the default
working directory in the course site. Locally, run from this chapter folder.
Uploading again to the same complete path replaces that input file.
Saved examples appear under `outputs/03_duplicates_and_sorting/` in the file browser.


## Select the Fields Needed for Cleaning

This example uses the original retail file independently. Select order
ID, date, and quantity, then rename them with a dictionary whose keys are
old column names and values are new names. The case has one row per order,
so a repeated order ID represents a duplicate record here.


In [1]:
import os

import pandas as pd

input_directory = os.path.join("assets", "03_duplicates_and_sorting")
output_directory = os.path.join("outputs", "03_duplicates_and_sorting")
input_path = os.path.join(input_directory, "retail_sales_raw.csv")
output_path = os.path.join(output_directory, "cleaned_orders.csv")
os.makedirs(output_directory, exist_ok=True)

raw_df = pd.read_csv(input_path, encoding="utf-8")
orders_df = raw_df[["Order ID", "Order Date", "Quantity"]].rename(columns={
    "Order ID": "order_id", "Order Date": "order_date", "Quantity": "quantity"
})
orders_df["order_date"] = pd.to_datetime(orders_df["order_date"], format="mixed")

print("Orders before cleaning:")
print(orders_df)
print("Initial row count:", len(orders_df))


Orders before cleaning:
    order_id order_date  quantity
0       1001 2026-01-03       2.0
1       1002 2026-01-05       5.0
2       1003 2026-01-07       1.0
3       1004 2026-01-12       1.0
4       1005 2026-01-18       NaN
5       1006 2026-02-02       3.0
6       1006 2026-02-02       3.0
7       1007 2026-02-08       2.0
8       1008 2026-02-10       1.0
9       1009 2026-02-16       2.0
10      1010 2026-02-21       2.0
11      1011 2026-03-03       3.0
Initial row count: 12


The table contains 12 rows. Order 1006 appears twice, and order 1005 has an unknown quantity.


## Inspect Repeated Keys before Removing Rows

`.duplicated(subset=["order_id"])` marks later occurrences of each key.
`keep=False` marks every occurrence so both records can be compared.
`.drop_duplicates(..., keep="first")` retains the first occurrence.
Confirm that repeated keys are accidental duplicates; real orders with
several line items would need a more specific key.


In [2]:
print("Extra occurrences:", orders_df.duplicated(subset=["order_id"]).sum())
print("All rows with repeated order IDs:")
print(orders_df.loc[orders_df.duplicated(subset=["order_id"], keep=False)])

unique_orders_df = orders_df.drop_duplicates(subset=["order_id"], keep="first").copy()
print("Rows after removing duplicates:", len(unique_orders_df))


Extra occurrences: 1
All rows with repeated order IDs:
   order_id order_date  quantity
5      1006 2026-02-02       3.0
6      1006 2026-02-02       3.0
Rows after removing duplicates: 11


There is one extra occurrence, so 11 rows remain. The displayed order 1006 records agree on date and quantity, which supports retaining one for this case.


## Fill, Sort, and Reset the Index

Compute the median after removing duplicates so a repeated record does
not receive extra weight. This example uses that median to estimate
the missing quantity. It equals a whole number here, allowing `.astype(int)`.
Do not use an integer cast to silently round a fractional estimate.

`.sort_values()` orders by date and then order ID to break ties.
`.reset_index(drop=True)` gives the result consecutive row labels.


In [3]:
median_quantity = unique_orders_df["quantity"].median()
unique_orders_df["quantity"] = unique_orders_df["quantity"].fillna(median_quantity).astype(int)
cleaned_df = unique_orders_df.sort_values(["order_date", "order_id"]).reset_index(drop=True)

print("Median quantity used:", median_quantity)
print("Cleaned and sorted orders:")
print(cleaned_df)


Median quantity used: 2.0
Cleaned and sorted orders:
    order_id order_date  quantity
0       1001 2026-01-03         2
1       1002 2026-01-05         5
2       1003 2026-01-07         1
3       1004 2026-01-12         1
4       1005 2026-01-18         2
5       1006 2026-02-02         3
6       1007 2026-02-08         2
7       1008 2026-02-10         1
8       1009 2026-02-16         2
9       1010 2026-02-21         2
10      1011 2026-03-03         3


The median is 2.0, so order 1005 receives quantity 2. The result has labels 0 through 10 and dates from January 3 to March 3, 2026. The estimate should not be mistaken for a measured quantity.


## Save the Retained Records

Save a new file and restore its date type on reload. The equality check
verifies the values, row order, column order, and types in this result.


In [4]:
cleaned_df.to_csv(output_path, index=False, encoding="utf-8", date_format="%Y-%m-%d")
reopened_df = pd.read_csv(output_path, encoding="utf-8", parse_dates=["order_date"])

print("Saved file:", output_path)
print("Reopened row count:", len(reopened_df))
print("Round trip matches:", reopened_df.equals(cleaned_df))
print("Duplicate keys left:", reopened_df.duplicated(subset=["order_id"]).sum())


Saved file: outputs/03_duplicates_and_sorting/cleaned_orders.csv
Reopened row count: 11
Round trip matches: True
Duplicate keys left: 0


The round trip matches, the saved file has 11 rows, and no order ID repeats. The supplied raw file remains unchanged.


Define what makes a record unique before removing duplicates. Estimate missing values from the retained records and sort with explicit tie-breaking columns.

[Removing duplicate rows](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)
